# 🏪 Spätis — Logic Refinement & POI Schema Alignment

- This notebook implements Step 2 (Logic Refinement & Execution) for the Spätis layer.
- Goal: transform OSM-based convenience store data into a POI-schema compliant dataset and prepare it for database loading.

## Transformation Overview

- Source: OpenStreetMap (shop=convenience, shop=kiosk)
- Column normalization (IDs, names, casing)
- POI schema alignment
- Spatial enrichment (district + neighborhood)
- Geometry → WKT POINT format
- Duplicate check (~10m proxy)
- Missing addresses enriched via Nominatim
- Final export for DB loading

# 1. Imports

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point

import requests
import os

from geopy.geocoders import Nominatim
from time import sleep
from tqdm import tqdm
tqdm.pandas()

# 2. Primary Source Extraction

In [2]:
# Overpass query for Spätis & corner stores in Berlin
query = """
[out:json];
(
  node[shop=convenience](52.4,13.2,52.7,13.6);
  node[shop=kiosk](52.4,13.2,52.7,13.6);
);
out body;
"""

# Send request
url = "https://overpass-api.de/api/interpreter"
resp = requests.post(url, data=query, timeout=180)
resp.raise_for_status()
data = resp.json()

# Normalize JSON to pandas dataframe
elements = data.get("elements", [])
df = pd.json_normalize(elements)

# Ensure lat/lon exist
df = df[df['lat'].notna() & df['lon'].notna()]

# Convert to GeoDataFrame
spatis_gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df['lon'].astype(float), df['lat'].astype(float))],
    crs="EPSG:4326"
)

# 3. Cleaning & Normalization
### 3.1 Rename Columns

In [3]:
# Minimal cleanup and renaming to match schema
spatis = spatis_gdf.rename(columns={
    'tags.name': 'name',
    'geometry': 'geometry', 
    'tags.addr:street' : 'address',
    'tags.opening_hours': 'opening_hours',
    'tags.phone': 'phone_number',
    'tags.website': 'website',
    'tags.email': 'email',
    'lat': 'latitude',
    'lon': 'longitude',
    'id': 'id' 
}).copy()

print("Records loaded:", len(spatis))
spatis.head()

Records loaded: 1616


,type,id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,address,tags.addr:suburb,...,tags.post_office:opening_hours,tags.money_transfer,tags.mobile,tags.fuel:HGV_diesel,tags.fuel:octane_100,tags.post_office:id_check,tags.name:ko,tags.public_transport,tags.ticket,geometry
0,node,26867411,52.501974,13.294496,Berlin,DE,14,10711,Heilbronner Straße,Halensee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.2945 52.50197)
1,node,29997723,52.508370,13.280947,Berlin,DE,8-10,14057,Messedamm,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.28095 52.50837)
2,node,63253672,52.499322,13.296118,Berlin,DE,39,10711,Joachim-Friedrich-Straße,Halensee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.29612 52.49932)
3,node,253616592,52.511047,13.462698,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.4627 52.51105)
4,node,266629404,52.509209,13.587500,Berlin,DE,45,12621,Mädewalder Weg,Kaulsdorf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.5875 52.50921)


### 3.2 ID Normalization 
OSM IDs are validated and converted to string format to match the
required VARCHAR(20) numeric-only POI schema.

In [4]:
spatis["id"] = spatis["id"].astype(str).copy()

# do not allow NULL values
if spatis["id"].isna().any(): 
    raise ValueError("Null IDs found")

is_numeric_id = spatis["id"].str.fullmatch(r"\d+")
invalid_ids = spatis["id"][~is_numeric_id]

# only allow numerical values
if not is_numeric_id.all():
    print(invalid_ids.head())
    raise ValueError("Non-numeric IDs found")

# only allow ids with 20 characters or less
if (spatis["id"].str.len() > 20).any():
    raise ValueError('ID with more than 20 characters')

### 3.3 Store Name Cleaning

Whitespace, formatting, and common spelling variants were normalized.
No semantic renaming of individual store names was performed.


In [5]:
spatis["name"].value_counts().head(10)

name
Spätkauf           67
Kiosk              27
REWE To Go         16
Späti              14
Lotto              13
ServiceStore DB    13
Mini Markt          9
Minimarkt           8
Kiez Shop           8
Spätshop            7
Name: count, dtype: int64

In [6]:
# 1) Baseline cleaning (whitespace & formatting)
spatis["name"] = (spatis["name"]
                  .astype(str)
                  .str.strip()
                  .str.replace(r"\s+", " ", regex=True)
)

# 2) simple and save replacement
repl = {
    "Mini Markt": "Minimarkt",
    "Mini-Markt": "Minimarkt",
    "Kiez Kiosk": "Kiezkiosk",
    "Asia-Markt": "Asia Markt",
    "latenight shop": "Late Night Shop",

}

spatis["name"] = spatis["name"].replace(repl)

In [7]:
#Capitalization only for purely generic names
generic_case = {
    "späti": "Späti",
    "spaeti": "Späti",
    "spati": "Späti",
    "spätkauf": "Spätkauf",
    "kiosk": "Kiosk",
}
spatis["name"] = spatis["name"].replace(generic_case)   

In [8]:
# fill missing + enforce schema length
spatis["name"] = spatis["name"].fillna("unknown_spati").str[:200]

# 4. Spatial Enrichment
Assign district and neighborhood information to each Späti using official Berlin boundary layers (LOR).

### 4.1 Load and Prepare District Boundaries
Load official district polygons, rename schema fields, and cast IDs to VARCHAR-compatible strings.


In [9]:
# Load districts
districts = gpd.read_file("../sources/bezirksgrenzen 2.geojson")

# Rename correctly
districts = districts.rename(columns={
    "Gemeinde_schluessel": "district_id",
    "Gemeinde_name": "district"
})

# Keep only the necessary columns
districts = districts[["district_id", "district", "geometry"]].copy()

# Convert types
districts["district_id"] = districts["district_id"].astype(str)

### 4.2 Spatial Join: Assign District to Each Späti
Each Späti point is spatially matched to a district polygon using a point-in-polygon join.


In [10]:
# Spatial join: assign each SPATIS point to a district
spatis = gpd.sjoin(
    spatis,
    districts[["district_id", "district", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

### 4.3 Load and Prepare Neighborhood (LOR) Boundaries
Load LOR neighborhood polygons, rename columns, align CRS, and prepare IDs.

In [11]:
# Load neighborhoods
neighborhoods = gpd.read_file("../sources/lor_ortsteile 2.geojson")

# Correct: rename first
neighborhoods = neighborhoods.rename(columns={
    "spatial_name": "neighborhood_id",      # ID
    "OTEIL": "neighborhood"            # name
})

# Now select the correct columns
neighborhoods = neighborhoods[["neighborhood_id", "neighborhood", "geometry"]].copy()

# Convert ID to string
neighborhoods["neighborhood_id"] = neighborhoods["neighborhood_id"].astype(str)

# CRS fix to match SPATIS
neighborhoods = neighborhoods.to_crs(spatis.crs)

print(neighborhoods.head())

  neighborhood_id  neighborhood  \
0            0101         Mitte   
1            0102        Moabit   
2            0103  Hansaviertel   
3            0104    Tiergarten   
4            0105       Wedding   

                                            geometry  
0  POLYGON ((13.41649 52.52696, 13.41635 52.52702...  
1  POLYGON ((13.33884 52.51974, 13.33884 52.51974...  
2  POLYGON ((13.34322 52.51557, 13.34323 52.51557...  
3  POLYGON ((13.36879 52.49878, 13.36891 52.49877...  
4  POLYGON ((13.34656 52.53879, 13.34664 52.53878...  


### 4.4 Spatial Join — Assign Neighborhood to Each Späti
Each Späti point is spatially matched to a LOR neighborhood polygon.

In [12]:
spatis = spatis.drop(columns=["index_right"], errors="ignore")

In [13]:
# Spatial join: neighborhoods

spatis = gpd.sjoin(
    spatis,
    neighborhoods[["neighborhood_id", "neighborhood", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

### 4.5 Spatial Join Validation - Missing IDs

In [14]:
spatis[["district_id", "district", "neighborhood_id", "neighborhood"]].isna().sum()


district_id        25
district           25
neighborhood_id    25
neighborhood       25
dtype: int64

### 4.6 Coordinate Boundary Check


In [15]:
# Coordinate validation: ensure all points fall within Berlin boundaries
invalid_coords = spatis[
    ~(
        spatis["latitude"].between(52.3, 52.7) &
        spatis["longitude"].between(13.0, 13.8)
    )
]

len(invalid_coords)

0

### 4.7 Geometry Conversion to WKT (POI Schema Compliance)

In [16]:
# geometry conversion: Geo → WKT (for export & DB)
spatis["geometry"] = spatis["geometry"].apply(
    lambda g: g.wkt if g is not None else None
)

/var/folders/v3/ygq50l_550q0rr8g2dzhhyfw0000gn/T/ipykernel_32586/451040415.py:2: UserWarning: Geometry column does not contain geometry.
  spatis["geometry"] = spatis["geometry"].apply(


In [17]:
len(spatis)

1616

In [18]:
spatis = spatis.dropna(subset=['district_id', 'district'])
print("After dropping missing districts, rows left:", len(spatis))

After dropping missing districts, rows left: 1591


## 5. Data Quality Checks

In [19]:
final_cols_for_export = [
    "id",
    "district_id",
    "neighborhood_id",
    "name",
    "address",
    "latitude",
    "longitude",
    "opening_hours",
    "district",
    "neighborhood",
    "geometry"]

df_export = spatis[final_cols_for_export]

In [20]:
df_export.info()

<class 'pandas.DataFrame'>
Index: 1591 entries, 0 to 1615
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1591 non-null   str    
 1   district_id      1591 non-null   str    
 2   neighborhood_id  1591 non-null   str    
 3   name             1591 non-null   str    
 4   address          824 non-null    str    
 5   latitude         1591 non-null   float64
 6   longitude        1591 non-null   float64
 7   opening_hours    714 non-null    str    
 8   district         1591 non-null   str    
 9   neighborhood     1591 non-null   str    
 10  geometry         1591 non-null   str    
dtypes: float64(2), str(9)
memory usage: 149.2 KB


### 5.1 Duplicate Detection
Duplicates checked using name + coordinate grouping (~10m tolerance proxy).

In [21]:
duplicates = (
    df_export
    .groupby([
        "name",
        spatis["latitude"].round(5),
        spatis["longitude"].round(5)
    ])
    .size()
    .loc[lambda x: x > 1]
)

In [22]:
print(duplicates)

name        latitude  longitude
The Market  52.50175  13.31299     2
dtype: int64


One duplicate “The Market” was identified based on name and ~10 m proximity.

In [23]:
len(df_export)

1591

In [24]:
df_export = df_export.drop_duplicates(
    subset=["name", "latitude", "longitude"]
)

In [25]:
len(df_export)

1590

### 5.2 Address Enrichment via Reverse Geocoding

In [26]:
missing_address_mask = df_export["address"].isna()
print("Missing addresses:", missing_address_mask.sum())

Missing addresses: 767


In [27]:
# Initialize the Nominatim geocoder with a custom user agent to avoid being blocked
geolocator = Nominatim(user_agent="berlin_spati_locator", timeout=10)


In [28]:
# reverse geocoding function that retrieves the address from latitude and longitude
def get_address(lat, lon):
    try:
        location = geolocator.reverse(
            (lat, lon), 
            exactly_one=True, 
            language='de')
        
        sleep(1)  # Respect Nominatim API limits
        
        if not location:
            return None
        
        addr = location.raw.get("address", {}) #address dictionary

        road = addr.get("road")
        square = addr.get("square")
        pedestrian = addr.get("pedestrian")
        house = addr.get("house_number")

        street_name = road or square or pedestrian

        if street_name and house:
            return f"{street_name} {house}"
        elif street_name:
            return street_name
        else:
            return None

    except Exception:
        return None

In [29]:
# use function for missing addresses
df_export.loc[missing_address_mask, "address"] = (
    df_export.loc[missing_address_mask]
    .progress_apply(
        lambda row: get_address(row["latitude"], row["longitude"]),
        axis=1
    )
)

100%|██████████| 767/767 [46:26<00:00,  3.63s/it]    


In [30]:
# check
print("Remaining missing addresses:",
      df_export["address"].isna().sum())

Remaining missing addresses: 0


### 5.3 District Mapping

In [40]:
df_export["district_id"].unique()


<StringArray>
['004', '002', '010', '012', '006', '001', '008', '009', '003', '007', '011',
 '005']
Length: 12, dtype: str

In [41]:
district_mapping = {
    "Mitte": "11001001",
    "Friedrichshain-Kreuzberg": "11002002",
    "Pankow": "11003003",
    "Charlottenburg-Wilmersdorf": "11004004",
    "Spandau": "11005005",
    "Steglitz-Zehlendorf": "11006006",
    "Tempelhof-Schöneberg": "11007007",
    "Neukölln": "11008008",
    "Treptow-Köpenick": "11009009",
    "Marzahn-Hellersdorf": "11010010",
    "Lichtenberg": "11011011",
    "Reinickendorf": "11012012"
}

df_export["district_id"] = df_export["district"].map(district_mapping)


In [31]:
df_export.info()

<class 'pandas.DataFrame'>
Index: 1590 entries, 0 to 1615
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1590 non-null   str    
 1   district_id      1590 non-null   str    
 2   neighborhood_id  1590 non-null   str    
 3   name             1590 non-null   str    
 4   address          1590 non-null   str    
 5   latitude         1590 non-null   float64
 6   longitude        1590 non-null   float64
 7   opening_hours    713 non-null    str    
 8   district         1590 non-null   str    
 9   neighborhood     1590 non-null   str    
 10  geometry         1590 non-null   str    
dtypes: float64(2), str(9)
memory usage: 149.1 KB


All required POI schema fields are present and validated.  
Column names and data types match the database contract.  
Data quality checks were applied and remaining records are complete and consistent.

The dataset is now POI-compliant and ready for export and database ingestion.

# 6. Export

In [ ]:
# --- Database connection settings ---
user_name = ''
password = ''
host = ''
port = ''
schema = ''
database = ''

# --- Imports ---
import os
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
from sqlalchemy.exc import IntegrityError, SQLAlchemyError

# --- Connect to the database using psycopg2 ---
conn = psycopg2.connect(
    host=host,
    port=port,
    dbname=database,
    user=user_name,
    password=password
)
conn.autocommit = True
cur = conn.cursor()

print("Connected successfully (psycopg2).")

Connected successfully (psycopg2).


In [43]:
# --- Drop table if exists ---
cur.execute(f"DROP TABLE IF EXISTS {schema}.spaetis;")
print("Dropped table spaetis (if existed).")

create_sql = f"""
CREATE TABLE {schema}.spaetis (
    id VARCHAR(20) PRIMARY KEY,
    district_id VARCHAR(20) NOT NULL,
    name VARCHAR(200) NOT NULL DEFAULT 'unknown_spati',
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),
    geometry VARCHAR NOT NULL,
    neighborhood VARCHAR(100),
    district VARCHAR(100),
    neighborhood_id VARCHAR(20) NOT NULL,
    address VARCHAR(200),
    opening_hours VARCHAR(300),
    CONSTRAINT district_id_fk FOREIGN KEY (district_id)
        REFERENCES {schema}.districts(district_id)
        ON DELETE RESTRICT ON UPDATE CASCADE
);
"""
cur.execute(create_sql)
print("Created new spaetis table.")

Dropped table spaetis (if existed).
Created new spaetis table.


In [ ]:
# --- Insert into the database using SQLAlchemy 
engine = create_engine(
    f'postgresql+psycopg2://{'username'}:{'passwort'}@{'host'}:{'port'}/{'database'}’)

try:
    df_export.to_sql(
        "spaetis",
        engine,
        schema=schema,
        if_exists="append",
        index=False
    )
    print("✅ Data successfully inserted into spaetis!")
except IntegrityError as ie:
    # Likely a FK violation or primary key conflict
    print("❌ IntegrityError while inserting data (possible FK or PK violation).")
    print(str(ie))
    raise
except SQLAlchemyError as sae:
    print("❌ SQLAlchemy error during insert:")
    print(str(sae))
    raise 

cur.close()
conn.close()


✅ Data successfully inserted into spaetis!
